In [1]:
import os
import shutil
from huggingface_hub import snapshot_download

# --- PASSO 1: PREPARAR O "TÚNEL" DE ARMAZENAMENTO ---

# Caminhos
pasta_pequena = '/kaggle/working/pretrained_model'
pasta_grande = '/kaggle/temp/pretrained_model'

# Limpeza: Se a pasta já existir no disco pequeno (e estiver corrompida), apaga ela
if os.path.exists(pasta_pequena) and not os.path.islink(pasta_pequena):
    shutil.rmtree(pasta_pequena)

# Cria a pasta real no disco grande (onde tem espaço de sobra)
os.makedirs(pasta_grande, exist_ok=True)

# Cria o Link Simbólico (O Pulo do Gato)
# Isso cria um "atalho" na pasta de trabalho que aponta para o temp
if not os.path.exists(pasta_pequena):
    os.symlink(pasta_grande, pasta_pequena)

print(f"Link criado! Tudo que for salvo em '{pasta_pequena}' irá para '{pasta_grande}'")

# --- PASSO 2: O DOWNLOAD ---

# Define o cache também para o temp (segurança extra)
os.environ['HF_HOME'] = '/kaggle/temp/cache'

model_repo_id = "allenai/OLMo-1B-0724-hf"

# Agora o download vai "cair" no link simbólico e encher o disco temporário, não o seu working dir
snapshot_download(repo_id=model_repo_id, local_dir='pretrained_model')

Link criado! Tudo que for salvo em '/kaggle/working/pretrained_model' irá para '/kaggle/temp/pretrained_model'


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.71G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

'/kaggle/temp/pretrained_model'

In [39]:
# Crie uma célula no topo e rode isso antes de qualquer outra coisa:
!git clone https://github.com/BernardoBelleza/llm-unlearning
%cd llm-unlearning
!pip install -r requirements.txt

Cloning into 'llm-unlearning'...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 69 (delta 0), reused 0 (delta 0), pack-reused 66 (from 4)
Receiving objects: 100% (69/69), 44.35 MiB | 42.01 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Encountered 4 file(s) that should have been pointers, but weren't:
	semeval25-unlearning-data/data/forget_train-00000-of-00001.parquet
	semeval25-unlearning-data/data/forget_validation-00000-of-00001.parquet
	semeval25-unlearning-data/data/retain_train-00000-of-00001.parquet
	semeval25-unlearning-data/data/retain_validation-00000-of-00001.parquet
/kaggle/working/llm-unlearning/llm-unlearning


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Sequential Unlearning with Gradient Difference (SUGD)

Gradient Difference implementation (with changes) from: [code](https://github.com/yaojin17/Unlearning_LLM)

Train on mixed chunks of retain and forget data. The loss comprises two terms, each of them corresponding to retain and forget samples. Loss for retain samples is weighed by a positive factor, while forget samples are weighed by -1. 

$$L = positive\_factor * L_{retain} + (-1) * L_{forget}$$

The chunks can be unbalanced, containing retain samples that are integer multiples of the forget chunk size or even the whole retain set. 

Parameters:

- **Sequential**: (True/False) Whether or not to perform sequential training. If false normal training is performed using all retain and forget data at once. 
- **Chunk size**: Size of the forget chunks
- **Split retain**: Whether or not to split the retain set in chunks as well. If True, their size will be determined by the positive ratio parameter.
- **Positive ratio**: Integer multiple for the number of retain samples. #retain_samples = positive_ratio * chunk_size
- **Positive factor**: Weight of the retain loss term, positive float number (default is 1)
- **Retain loss**: Currently only Cross-entropy (“CE”) is implemented.

## Imports

In [3]:
import warnings
import torch
import json
import time
import gc
import os

from peft import LoraConfig, get_peft_model, TaskType
from huggingface_hub import snapshot_download
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments
)
from utils.data import DatasetProcessor
from utils.utils import (
    make_compute_metrics,
    preprocess_logits_for_metrics,
    print_number_of_trainable_model_parameters,
    print_gpu_memory,
    plot_metrics,
    plot_training_stats
)
from utils.evaluation import (
    QualitativeEvaluation,
    QuantitativeEvaluation,
    MMLU
)
from methods import AscentPlusDescentDataCollator, SequentialUnlearning

warnings.filterwarnings('ignore')

2026-02-09 23:45:04.675940: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770680704.855277      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770680704.914407      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770680705.372841      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770680705.372876      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770680705.372879      55 computation_placer.cc:177] computation placer alr

## Load training parameters

In [ ]:
# --- 1. CONFIGURAÇÃO (SemEval-2025 Task 4) ---
print("Configurando Hiperparâmetros...")
output_dir = 'output_dir'
os.makedirs(output_dir, exist_ok=True)

# Carrega base do JSON e sobrescreve com valores desejados
with open("configs/sequential_args.json", 'r') as f:
    args = json.load(f)

# Modelo
args["model_params"]["model_size"] = "1B"
args["model_params"]["torch_dtype"] = "float32"

# LoRA
args["model_params"]["apply_lora"] = True
args["model_params"]["train_last_k"] = False
args["model_params"]["lora_r"] = 16
args["model_params"]["lora_alpha"] = 64
args["model_params"]["lora_dropout"] = 0.05
args["model_params"]["target_modules"] = ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"]

# Dados
args["general"]["chunk_size"] = 32
args["general"]["positive_ratio"] = 2

# Treinamento
args["training_args"]["per_device_train_batch_size"] = 1
args["training_args"]["gradient_accumulation_steps"] = 4
args["training_args"]["gradient_checkpointing"] = True
args["training_args"]["learning_rate"] = 1e-5
args["training_args"]["num_epochs"] = 1

# Salva config
with open(f"{output_dir}/training_args.json", "w") as f:
    f.write(json.dumps(args, indent=4))
print("Configuracao aplicada!")

# --- 2. CARREGAMENTO DO MODELO ---
print("\nCarregando Modelo 1B...")
model_name = "AllenAI/OLMo-1B-hf" if args["model_params"]["model_size"] == "1B" else "AllenAI/OLMo-7B-hf"

try:
    model = AutoModelForCausalLM.from_pretrained(
        "pretrained_model",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    tokenizer = AutoTokenizer.from_pretrained("pretrained_model")
except:
    print(f"Pasta local nao encontrada. Baixando {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Modelo carregado. Memoria: {model.get_memory_footprint() / 1e9:.2f} GB")

# --- 3. APLICANDO LORA ---
print("\nAplicando LoRA...")
lora_config = LoraConfig(
    r=args["model_params"]["lora_r"],
    lora_alpha=args["model_params"]["lora_alpha"],
    target_modules=args["model_params"]["target_modules"],
    lora_dropout=args["model_params"]["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)

if args["training_args"]["gradient_checkpointing"]:
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()

model.print_trainable_parameters()
print("\nPronto para treinar!")

⚙️ Configurando Hiperparâmetros do Paper...
✅ Configuração aplicada!

📥 Carregando Modelo 1B...
Alvo: AllenAI/OLMo-1B-hf
⚠️ Pasta local não encontrada. Baixando AllenAI/OLMo-1B-hf...


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.71G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Modelo carregado. Memória usada: 4.71 GB (Esperado: ~4.5 GB)

🔧 Aplicando LoRA...
trainable params: 12,058,624 || all params: 1,188,823,040 || trainable%: 1.0143

🚀 TUDO PRONTO! Pode rodar o trainer.train() agora.


In [5]:
# Verifica onde o modelo está agora (Provavelmente dirá 'cpu')
print(f"O modelo está no dispositivo: {model.device}")

# Força o envio para a GPU
print("Movendo modelo para GPU...")
model.to("cuda")

# Verifica novamente a memória (Agora deve mostrar ~4.5GB ocupados)
print(f"Novo dispositivo: {model.device}")
print_gpu_memory()

O modelo está no dispositivo: cpu
Movendo modelo para GPU...
Novo dispositivo: cuda:0

GPU 0: 4536.00 MB allocated
GPU 0: 4538.00 MB cached

GPU 1: 0.00 MB allocated
GPU 1: 0.00 MB cached


In [6]:
start = time.time()

In [10]:
print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 12058624
all model parameters: 1188823040
percentage of trainable model parameters: 1.01%


In [11]:
print_gpu_memory()


GPU 0: 4536.00 MB allocated
GPU 0: 4538.00 MB cached

GPU 1: 0.00 MB allocated
GPU 1: 0.00 MB cached


## Prepare Data

In [12]:
# Initialize
processor = DatasetProcessor(data_dir='semeval25-unlearning-data/data', tokenizer=tokenizer, n_samples_per_task=None)

# Construct the tokenized datasets as a DatasetDict
dataset = processor(split=args["general"]["split"], task='all', split_tasks=False, split_retain=False)

# Define the data collator
data_collator = AscentPlusDescentDataCollator(tokenizer=tokenizer, padding='longest', pad_to_multiple_of=8)

Map:   0%|          | 0/1136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1112 [00:00<?, ? examples/s]

In [13]:
dataset

DatasetDict({
    retain: Dataset({
        features: ['id', 'input', 'output', 'task', 'split', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1136
    })
    forget: Dataset({
        features: ['id', 'input', 'output', 'task', 'split', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1112
    })
})

## Training Setup

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=args["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=args["training_args"]["gradient_accumulation_steps"],
    gradient_checkpointing=args["training_args"]["gradient_checkpointing"],
    per_device_eval_batch_size=16,
    eval_accumulation_steps=1,
    learning_rate=args["training_args"]["learning_rate"],
    num_train_epochs=args["training_args"]["num_epochs"],
    logging_steps=4,
    save_strategy="no",
    eval_strategy="no",
    fp16=True,
    report_to="none",
    include_inputs_for_metrics=True
)

trainer = SequentialUnlearning(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    training_args=training_args,
    forget_dataset=dataset['forget'],
    retain_dataset=dataset['retain'],
    compute_metrics=make_compute_metrics(model, tokenizer, max_samples=32),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    sequential=args["general"]["sequential"],
    chunk_size=args["general"]["chunk_size"],
    positive_ratio=args["general"]["positive_ratio"],
    positive_factor=args["general"]["positive_factor"],
    retain_loss=args["general"]["retain_loss"]
)

Using `include_inputs_for_metrics` is deprecated and will be removed in version 5 of 🤗 Transformers. Please use `include_for_metrics` list argument instead.


## Train

In [18]:
from methods.sequential import AscentPlusDescentTrainer

# O TRUQUE ANTI-RECURSÃO:
# Verificamos se já salvamos a original antes. Se sim, não fazemos nada.
if not hasattr(AscentPlusDescentTrainer, "_original_get_train_sampler"):
    
    # 1. Salva a original numa variável escondida dentro da própria classe
    AscentPlusDescentTrainer._original_get_train_sampler = AscentPlusDescentTrainer._get_train_sampler

    # 2. Define a nova função que usa essa cópia escondida
    def novo_sampler(self, dataset=None):
        return AscentPlusDescentTrainer._original_get_train_sampler(self)

    # 3. Aplica o patch
    AscentPlusDescentTrainer._get_train_sampler = novo_sampler
    print("✅ Correção aplicada com sucesso!")

else:
    print("⚡ Correção já estava ativa (ignorando para evitar erro de recursão).")

✅ Correção aplicada com sucesso!


In [20]:
trainer.train(split_retain=args["general"]["split_retain"])

Number of chunks: 34
Remaining forget samples: 24

Training on chunk 1 ...


Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 50279}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss


KeyboardInterrupt: 

In [37]:
trainer.save_model()

NameError: name 'trainer' is not defined

In [38]:
summary = trainer.save_summary(output_dir)

print(summary['total_runtime'])
print(summary['total_flos'])

NameError: name 'trainer' is not defined

In [24]:
plot_metrics(summary["log_history"], output_dir)

KeyError: "['eval_retain_1_runtime', 'eval_retain_1_samples_per_second', 'eval_retain_1_steps_per_second'] not found in axis"

In [ ]:
plot_training_stats(summary["log_history"])

In [ ]:
if args["model_params"]["apply_lora"]:
    model = model.merge_and_unload()
    print(model)

NameError: name 'model' is not defined

In [ ]:
# Salva o modelo merged completo (não apenas adapter LoRA)
# Assim MMLU e QuantitativeEvaluation conseguem carregar com AutoModelForCausalLM.from_pretrained()
model.save_pretrained("unlearned_model")
tokenizer.save_pretrained("unlearned_model")

# Também salva em output_dir/final_model como modelo completo para as avaliações
model.save_pretrained(f"{output_dir}/final_model")
tokenizer.save_pretrained(f"{output_dir}/final_model")

In [ ]:
end = time.time()
print("Total training time:", end-start)

In [ ]:
print_gpu_memory()

In [25]:
del trainer
del model

gc.collect()
torch.cuda.empty_cache()

In [26]:
print_gpu_memory()


GPU 0: 16.25 MB allocated
GPU 0: 18.00 MB cached

GPU 1: 16.25 MB allocated
GPU 1: 20.00 MB cached


## Final Evaluation

In [46]:
# Optionally run evaluation on MMLU first
# The MMLU class doesn't run the code of the official MMLU repo.
# However, it provides correct results.

# The list of all 57 topics. Choose a subset for faster results
topics = ['abstract_algebra',
          'anatomy',
          'astronomy',
          'business_ethics',
          'clinical_knowledge',
          'college_biology',
          'college_chemistry',
          'college_computer_science',
          'college_mathematics',
          'college_medicine',
          'college_physics',
          'computer_security',
          'conceptual_physics',
          'econometrics',
          'electrical_engineering',
          'elementary_mathematics',
          'formal_logic',
          'global_facts',
          'high_school_biology',
          'high_school_chemistry',
          'high_school_computer_science',
          'high_school_european_history',
          'high_school_geography',
          'high_school_government_and_politics',
          'high_school_macroeconomics',
          'high_school_mathematics',
          'high_school_microeconomics',
          'high_school_physics',
          'high_school_psychology',
          'high_school_statistics',
          'high_school_us_history',
          'high_school_world_history',
          'human_aging',
          'human_sexuality',
          'international_law',
          'jurisprudence',
          'logical_fallacies',
          'machine_learning',
          'management',
          'marketing',
          'medical_genetics',
          'miscellaneous',
          'moral_disputes',
          'moral_scenarios',
          'nutrition',
          'philosophy',
          'prehistory',
          'professional_accounting',
          'professional_law',
          'professional_medicine',
          'professional_psychology',
          'public_relations',
          'security_studies',
          'sociology',
          'us_foreign_policy',
          'virology',
          'world_religions']

mmlu_start = time.time()
mmlu = MMLU(topics)
mmlu.run(model_path="output_dir/final_model", mmlu_metrics_file_path=f"{output_dir}/evaluation/mmlu.json")

print("MMLU time: ", time.time()-mmlu_start)

OSError: None is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [47]:
# Quantitative Evaluation

evaluation_args = {
    "seed": 42,
    "debug": True,
    "keep_files": True,
    "max_new_tokens": 256,
    "compute_metrics_only": False,
    "batch_size": 8,
    "mia_data_path": "semeval25-unlearning-data/mia_data/",
    "split": args["general"]["split"],
    "data_path": "semeval25-unlearning-data/data/",

    "checkpoint_path": "output_dir/final_model",
    
    # "checkpoint_path": "unlearned_model",
    "output_dir": f"{output_dir}/evaluation",
    "mmlu_metrics_file_path": f"{output_dir}/evaluation/mmlu.json"
}

quantitative_eval = QuantitativeEvaluation(evaluation_args)

In [48]:
quantitative_eval.run()

Evaluating Checkpoint at output_dir/final_model


OSError: None is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Qualitative evaluation

qualitative_eval = QualitativeEvaluation(
    checkpoint_path="unlearned_model",
    path_to_predictions=f'{output_dir}/evaluation',
    path_to_gqa='utils/general_questions.json',
    output_dir=f'{output_dir}/evaluation/qualitative',
    n_samples=5
)

In [ ]:
qualitative_eval.run()